# C9. Speed

Is the world fast enough to explore? The final world against its distilled students.

See `docs/plans/experiments-plan.md`, entry C9. Results are read from `artifacts/results/C9/`. Nothing here trains or samples.

## Run

Run these from the repo root, in order. `uv run erosion-sim list` shows what is already trained.

**1. Data.** Skip any that already exist.

```sh
sbatch slurm/datagen.sbatch main_512
```

**2. Train.** One command per model; each resumes if stopped.

```sh
uv run erosion-sim train C9 coarse_student
uv run erosion-sim train C9 fine_student
```

- `coarse` is `C6/coarse_seed0`, trained in C6. If it is not trained yet, `uv run erosion-sim train C9 coarse` trains it there.
- `fine` is `C6/fine_seed0`, trained in C6. If it is not trained yet, `uv run erosion-sim train C9 fine` trains it there.
- `coarse_student` needs `C6/coarse_seed0` trained first (teacher).
- `fine_student` needs `C6/fine_seed0` trained first (teacher).

**3. Evaluate.**

```sh
uv run erosion-sim evaluate C9
```

On the cluster, `uv run erosion-sim submit C9` does steps 2 and 3 as Slurm jobs.

Then run the cells below.

In [ ]:
import json

from erosion_sim.erosion import plots
from erosion_sim.experiments import registry
from erosion_sim.experiments.results import read_results
from erosion_sim.experiments.runner import model_folder
from erosion_sim.training.train import plot_history

NAME = "C9"
exp = registry.get(NAME)
results = read_results(NAME)
print(exp.question)
{"models": {label: m["folder"] for label, m in results.manifest["models"].items()},
 **{k: results.manifest[k] for k in ("code", "time", "slurm_job")}}

## Training

In [ ]:
for label in exp.models:
    folder = model_folder(NAME, label)[0]
    print(label, folder)
    plot_history(json.loads((folder / "history.json").read_text()))

## Worlds, levels alone, and baselines, on the runs' interiors

In [ ]:
plots.table(results, plots.TABLE + ('chunk_edge_rmse', 'seconds_per_query'))

In [ ]:
if "dT" in results.scalars["by"]:
    plots.by_table(results, "dT", "skill", kinds=("world", "baseline"))

## Seams

How much neighboring windows disagree before blending, per level and phase, as a share of the change.

In [ ]:
for world, levels in results.scalars["seams"].items():
    for level, phases in levels.items():
        print(world, level, [f"{p['relative']:.1%}" for p in phases])

## Examples

In [ ]:
plots.plot_examples(results, kinds=("world", "baseline"))

## Conclusion

_Written after reading the results above._